# Model-assisted audit near-field
- Mục đích: tạo queue để bạn duyệt nhãn dẫn xuất; không train, không sửa dataset gốc.
- DeepQuest: nhãn chỉ cấp ảnh. Teacher đưa bbox gợi ý; chỉ ảnh bạn bấm “Duyệt bbox model” mới có bbox trong dataset dẫn xuất.
- FireAndSmoke v1: đưa toàn bộ 100 ảnh vào queue để xác định có thực sự chỉ là `fire` hay không.
- Notebook tự chứa toàn bộ code. Kaggle Input chỉ có dữ liệu ảnh/nhãn và checkpoint nhị phân.
- Dùng tối đa 2 GPU T4 độc lập; output `near_field_model_audit_bundle.zip`.
- Đỏ liền = nhãn gốc fire; xanh liền = nhãn gốc smoke; nét đứt = teacher.


In [ ]:
from pathlib import Path
DATA_ROOT = Path('/kaggle/input/near-field-audit-data')
CHECKPOINT = Path('/kaggle/input/rfdetr-large-dfire-winner/checkpoint_best_total.pth')
OUTPUT_ROOT = Path('/kaggle/working/near_field_model_audit')
assert DATA_ROOT.is_dir(), DATA_ROOT
assert CHECKPOINT.is_file(), CHECKPOINT
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'data_root': str(DATA_ROOT), 'checkpoint': str(CHECKPOINT), 'output': str(OUTPUT_ROOT)})


In [ ]:
import subprocess, sys, torch
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rfdetr==1.8.3'], check=True)
GPU_COUNT = min(2, torch.cuda.device_count())
assert GPU_COUNT >= 1, 'Kaggle chưa bật GPU accelerator'
print({'gpus_used': GPU_COUNT, 'names': [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]})


In [ ]:
from pathlib import Path
WORKER = Path('/kaggle/working/near_field_audit_worker.py')
WORKER.write_text('import argparse\nimport json\nimport os\nfrom collections import defaultdict\nfrom pathlib import Path\n\nSOURCES = {\n    "dfs_fire_v3": {"root": "dfs_fire_v3", "splits": ("train", "valid", "test"), "labels": None},\n    "home_fire_2025": {"root": "home_fire_2025", "splits": ("train", "val", "test"), "labels": None},\n    "indoor_fire_smoke_2025": {"root": "indoor_fire_smoke_2025/Indoor Fire Smoke", "splits": ("train", "valid", "test"), "labels": None},\n    "annotated_fire_smoke_2025": {"root": "annotated_fire_smoke_2025", "splits": ("train", "valid", "test"), "labels": None},\n    "fasdd_cv_v9": {"root": "fasdd_cv_v9/FASDD_CV", "splits": ("",), "labels": "annotations/YOLO_CV/labels"},\n    "fire_and_smoke_v1": {"root": "fire_and_smoke_v1", "splits": ("train", "valid", "test"), "labels": None, "force_review": True, "class_map": {"0": "fire"}},\n    "deepquest_ai": {"root": "deepquest_ai/FIRE-SMOKE-DATASET", "kind": "image_classification", "splits": ("Train", "Test")},\n}\n\ndef args():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data-root\', required=True)\n    parser.add_argument(\'--weights\', required=True)\n    parser.add_argument(\'--gpu\', required=True)\n    parser.add_argument(\'--rank\', type=int, required=True)\n    parser.add_argument(\'--world-size\', type=int, required=True)\n    parser.add_argument(\'--output\', required=True)\n    parser.add_argument(\'--batch\', type=int, default=8)\n    parser.add_argument(\'--imgsz\', type=int, default=1280)\n    parser.add_argument(\'--conf\', type=float, default=0.05)\n    parser.add_argument(\'--missing-conf\', type=float, default=0.70)\n    parser.add_argument(\'--unsupported-conf\', type=float, default=0.10)\n    parser.add_argument(\'--per-source-buffer\', type=int, default=160)\n    return parser.parse_args()\n\ndef label_path(image, root, config):\n    if config[\'labels\']:\n        return root / config[\'labels\'] / f\'{image.stem}.txt\'\n    return image.parent.parent / \'labels\' / f\'{image.stem}.txt\'\n\ndef records(data_root):\n    result = []\n    for source, config in SOURCES.items():\n        root = data_root / config[\'root\']\n        if not root.is_dir():\n            raise FileNotFoundError(root)\n        if config.get(\'kind\') == \'image_classification\':\n            for split in config[\'splits\']:\n                for class_dir in sorted((root / split).iterdir()):\n                    if not class_dir.is_dir() or class_dir.name not in {\'Fire\', \'Smoke\', \'Neutral\'}:\n                        continue\n                    image_class = class_dir.name.lower()\n                    for image in sorted(class_dir.glob(\'*\')):\n                        if image.suffix.lower() in {\'.jpg\', \'.jpeg\', \'.png\'}:\n                            result.append({\'source\': source, \'source_split\': split, \'image\': str(image), \'label\': None, \'dataset_boxes\': [], \'label_classes\': [] if image_class == \'neutral\' else [image_class], \'source_label_kind\': \'image_level\', \'source_image_class\': image_class, \'force_review\': False})\n            continue\n        class_map = config.get(\'class_map\', {\'0\': \'fire\', \'1\': \'smoke\'})\n        for split in config[\'splits\']:\n            images = root / split / \'images\' if split else root / \'images\'\n            for image in sorted(images.glob(\'*\')):\n                if image.suffix.lower() not in {\'.jpg\', \'.jpeg\', \'.png\'}:\n                    continue\n                label = label_path(image, root, config)\n                boxes = []\n                if label.exists():\n                    for line in label.read_text(encoding=\'utf-8\').splitlines():\n                        values = line.split()\n                        if len(values) == 5 and values[0] in class_map:\n                            boxes.append({\'class\': class_map[values[0]], \'center_x\': float(values[1]), \'center_y\': float(values[2]), \'width\': float(values[3]), \'height\': float(values[4])})\n                result.append({\'source\': source, \'source_split\': split or \'unsplit\', \'image\': str(image), \'label\': str(label), \'dataset_boxes\': boxes, \'label_classes\': sorted({box[\'class\'] for box in boxes}), \'source_label_kind\': \'bbox\', \'force_review\': config.get(\'force_review\', False)})\n    return result\n\ndef model_map(names):\n    values = list(names.values()) if isinstance(names, dict) else list(names)\n    lowered = [str(value).lower() for value in values]\n    smoke = next((index for index, value in enumerate(lowered) if \'smoke\' in value), None)\n    fire = next((index for index, value in enumerate(lowered) if \'fire\' in value), None)\n    if smoke is None or fire is None:\n        raise ValueError(f\'Teacher needs smoke and fire classes, got {values}\')\n    return smoke, fire\n\ndef main():\n    config = args()\n    os.environ[\'CUDA_VISIBLE_DEVICES\'] = config.gpu\n    from rfdetr import RFDETRLarge\n    all_records = records(Path(config.data_root))\n    shard = all_records[config.rank::config.world_size]\n    model = RFDETRLarge.from_checkpoint(config.weights)\n    smoke_id, fire_id = model_map(model.class_names)\n    candidates = defaultdict(list)\n    for start in range(0, len(shard), config.batch):\n        batch = shard[start:start + config.batch]\n        detections = model.predict([record[\'image\'] for record in batch], threshold=config.conf, shape=(config.imgsz, config.imgsz), include_source_image=False)\n        if not isinstance(detections, list):\n            detections = [detections]\n        for record, detection in zip(batch, detections):\n            scores = {\'smoke\': 0.0, \'fire\': 0.0}\n            teacher_boxes = []\n            for class_id, confidence, xyxy in zip(detection.class_id, detection.confidence, detection.xyxy):\n                label = \'smoke\' if int(class_id) == smoke_id else \'fire\' if int(class_id) == fire_id else None\n                if label:\n                    confidence = float(confidence)\n                    scores[label] = max(scores[label], confidence)\n                    teacher_boxes.append({\'class\': label, \'confidence\': confidence, \'xyxy\': [float(value) for value in xyxy]})\n            expected = set(record[\'label_classes\'])\n            predicted = {name for name, score in scores.items() if score >= config.missing_conf}\n            missing = sorted(predicted - expected)\n            unsupported = sorted(name for name in expected if scores[name] <= config.unsupported_conf)\n            kind, classes = ((\'possible_missing_label\', missing) if missing else (\'possible_wrong_or_extra_label\', unsupported) if unsupported else (None, []))\n            if record[\'source_label_kind\'] == \'image_level\' and record[\'source_image_class\'] in {\'fire\', \'smoke\'} and scores[record[\'source_image_class\']] >= config.missing_conf:\n                kind, classes = \'possible_pseudo_label\', [record[\'source_image_class\']]\n            if record[\'force_review\']:\n                kind, classes = \'full_source_semantic_review\', record[\'label_classes\'] or [\'fire\']\n            if kind:\n                candidates[record[\'source\']].append({**record, \'issue\': kind, \'classes_to_check\': classes, \'teacher_scores\': scores, \'teacher_boxes\': teacher_boxes, \'priority\': max(scores[name] for name in classes)})\n    selected = []\n    for source, items in candidates.items():\n        selected.extend(sorted(items, key=lambda item: item[\'priority\'], reverse=True)[:config.per_source_buffer])\n    Path(config.output).write_text(json.dumps({\'rank\': config.rank, \'world_size\': config.world_size, \'records_scanned\': len(shard), \'records\': selected}, ensure_ascii=False), encoding=\'utf-8\')\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
print(WORKER)


In [ ]:
import os, subprocess, time
jobs = []
for rank in range(GPU_COUNT):
    output = OUTPUT_ROOT / f'worker_{rank}.json'
    command = [sys.executable, str(WORKER), '--data-root', str(DATA_ROOT), '--weights', str(CHECKPOINT), '--gpu', str(rank), '--rank', str(rank), '--world-size', str(GPU_COUNT), '--output', str(output)]
    jobs.append(subprocess.Popen(command))
while any(job.poll() is None for job in jobs):
    subprocess.run(['nvidia-smi', '--query-gpu=index,utilization.gpu,memory.used,memory.total', '--format=csv,noheader'], check=False)
    time.sleep(30)
assert all(job.returncode == 0 for job in jobs), [job.returncode for job in jobs]
print('Inference xong.')


In [ ]:
import json, shutil
from collections import defaultdict
workers = [json.loads(path.read_text(encoding='utf-8')) for path in sorted(OUTPUT_ROOT.glob('worker_*.json'))]
grouped = defaultdict(list)
for worker in workers:
    for record in worker['records']:
        grouped[record['source']].append(record)
records = []
for source, items in sorted(grouped.items()):
    limit = 100 if source != 'deepquest_ai' else 100
    records.extend(sorted(items, key=lambda item: item['priority'], reverse=True)[:limit])
for index, record in enumerate(records, 1):
    record['review_id'] = f'model-audit-{index:05d}'
bundle = OUTPUT_ROOT / 'near_field_model_audit_bundle'
images = bundle / 'images'
images.mkdir(parents=True, exist_ok=True)
for record in records:
    source = Path(record['image'])
    target = images / f"{record['review_id']}{source.suffix.lower()}"
    shutil.copy2(source, target)
    record['bundle_image'] = target.relative_to(bundle).as_posix()
queue = {'purpose': 'Teacher boxes become derived labels only after explicit human approval; raw datasets stay unchanged.', 'records_selected': len(records), 'records': records}
(bundle / 'queue.json').write_text(json.dumps(queue, indent=2, ensure_ascii=False), encoding='utf-8')
payload = json.dumps(records, ensure_ascii=False).replace('</', '<\/')
page = '<!doctype html><meta charset="utf-8"><title>Near-field audit</title><style>body{font:16px system-ui;margin:24px;max-width:1200px}button,label{font:inherit;margin:4px;padding:10px}canvas{display:block;max-width:100%;border:1px solid #999;margin:12px 0}textarea{width:100%;height:70px}</style><h1>Near-field model-assisted review</h1><p>Đỏ liền = bbox nhãn fire; xanh liền = bbox nhãn smoke. Nét đứt = model. Bấm “Duyệt bbox model” chỉ khi bạn xác nhận bbox nét đứt đúng; nhãn đó chỉ được dùng trong dataset dẫn xuất, không sửa dữ liệu gốc.</p><p id=meta></p><canvas id=image></canvas><div><button onclick=pass()>Nhãn đúng [1]</button><label><input id=missing_or_extra type=checkbox onchange=issue(\'missing_or_extra\')> Thiếu/thừa bbox [2]</label><label><input id=wrong_class type=checkbox onchange=issue(\'wrong_class\')> Sai class bbox [3]</label><label><input id=geometry type=checkbox onchange=issue(\'geometry\')> Bbox sai vị trí/kích thước [4]</label><button onclick=uncertain()>Không chắc [5]</button><button onclick=adopt()>Duyệt bbox model làm nhãn dẫn xuất [6]</button><button onclick=exclude()>Loại ảnh khỏi train [7]</button></div><textarea id=note placeholder="Ghi chú ngắn nếu cần"></textarea><p><button onclick=prev()>Trước</button><button onclick=next()>Sau</button><button onclick=download()>Tải kết quả JSON</button></p><script>const records=__PAYLOAD__,key=\'near_field_model_audit_state\',state=JSON.parse(localStorage.getItem(key)||\'{}\'),canvas=document.querySelector(\'#image\'),ctx=canvas.getContext(\'2d\'),note=document.querySelector(\'#note\'),issues=[\'missing_or_extra\',\'wrong_class\',\'geometry\'];let i=0;function current(){return state[records[i].review_id]||{status:\'pending\',issues:[],label_action:\'keep_dataset\',note:\'\'}}function save(){state[records[i].review_id]={...current(),note:note.value};localStorage.setItem(key,JSON.stringify(state))}function draw(){let r=records[i],s=current();note.value=s.note||\'\';issues.forEach(x=>document.querySelector(\'#\'+x).checked=s.issues.includes(x));let original=r.source_label_kind===\'image_level\'?`nhãn cấp ảnh: ${r.source_image_class}`:`bbox gốc: ${r.label_classes.join(\',\')||\'empty\'}`;document.querySelector(\'#meta\').textContent=`${i+1}/${records.length} · ${r.source} · gợi ý: ${r.issue} (${r.classes_to_check.join(\',\')}) · ${original}`;let im=new Image();im.onload=()=>{canvas.width=im.naturalWidth;canvas.height=im.naturalHeight;ctx.drawImage(im,0,0);let w=Math.max(2,im.naturalWidth/450);for(let b of r.dataset_boxes){ctx.setLineDash([]);ctx.strokeStyle=b.class===\'fire\'?\'#ff3030\':\'#00a8ff\';ctx.lineWidth=w;ctx.strokeRect((b.center_x-b.width/2)*canvas.width,(b.center_y-b.height/2)*canvas.height,b.width*canvas.width,b.height*canvas.height)}for(let b of r.teacher_boxes){ctx.setLineDash([w*3,w*2]);ctx.strokeStyle=b.class===\'fire\'?\'#9b0000\':\'#005b8c\';ctx.lineWidth=w;ctx.strokeRect(b.xyxy[0],b.xyxy[1],b.xyxy[2]-b.xyxy[0],b.xyxy[3]-b.xyxy[1]);ctx.fillStyle=ctx.strokeStyle;ctx.fillText(`${b.class} ${b.confidence.toFixed(2)}`,b.xyxy[0],Math.max(14,b.xyxy[1]-3))}ctx.setLineDash([])};im.src=r.bundle_image}function pass(){state[records[i].review_id]={...current(),status:\'pass\',issues:[],label_action:\'keep_dataset\'};save();next()}function uncertain(){state[records[i].review_id]={...current(),status:\'uncertain\',issues:[],label_action:\'needs_manual\'};save();next()}function adopt(){let r=records[i];if(!r.teacher_boxes.length){alert(\'Model không có bbox để duyệt.\');return}state[r.review_id]={...current(),status:\'issue\',issues:[...new Set([...current().issues,\'missing_or_extra\'])],label_action:\'adopt_teacher\'};save();next()}function exclude(){let r=records[i];state[r.review_id]={...current(),status:\'issue\',issues:[...new Set([...current().issues,\'missing_or_extra\'])],label_action:\'exclude_from_training\'};save();next()}function issue(x){let s=current(),v=new Set(s.issues);v.has(x)?v.delete(x):v.add(x);state[records[i].review_id]={...s,status:v.size?\'issue\':\'pending\',issues:[...v]};save()}function next(){save();i=Math.min(i+1,records.length-1);draw()}function prev(){save();i=Math.max(i-1,0);draw()}function download(){save();let x=records.map(r=>({review_id:r.review_id,image:r.image,source:r.source,...(state[r.review_id]||{status:\'pending\',issues:[],label_action:\'keep_dataset\',note:\'\'})})),a=document.createElement(\'a\');a.href=URL.createObjectURL(new Blob([JSON.stringify(x,null,2)],{type:\'application/json\'}));a.download=\'near_field_model_audit_decisions.json\';a.click()}addEventListener(\'keydown\',e=>{if(e.key===\'1\')pass();if(e.key===\'2\')issue(\'missing_or_extra\');if(e.key===\'3\')issue(\'wrong_class\');if(e.key===\'4\')issue(\'geometry\');if(e.key===\'5\')uncertain();if(e.key===\'6\')adopt();if(e.key===\'7\')exclude();if(e.key===\'ArrowRight\')next();if(e.key===\'ArrowLeft\')prev()});draw()</script>'.replace('__PAYLOAD__', payload)
(bundle / 'review.html').write_text(page, encoding='utf-8')
archive = shutil.make_archive(str(OUTPUT_ROOT / 'near_field_model_audit_bundle'), 'zip', OUTPUT_ROOT, bundle.name)
print({'selected': len(records), 'bundle': archive})
